# Analyze Anime ratings and provide recommendation

In [1]:
import pandas as pd
df = pd.read_csv("imdb_anime.csv")
df.head()

,Title,Genre,User Rating,Number of Votes,Runtime,Year,Summary,Stars,Certificate,Metascore,Gross,Episode,Episode Title
0,One Piece,"Animation, Action, Adventure",8.9,"187,689",24 min,(1999– ),Follows the adventures of Monkey D. Luffy and ...,"Mayumi Tanaka,Laurent Vernin,Akemi Okamura,Ton...",TV-14,NaN,187689,0,NaN
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,"Animation, Action, Adventure",7.4,"28,895",99 min,(2023),The film follows the Turtle brothers as they w...,NaN,PG,74,28895,0,NaN
2,The Super Mario Bros. Movie,"Animation, Adventure, Comedy",7.1,"189,108",92 min,(2023),A plumber named Mario travels through an under...,NaN,PG,46,189108,0,NaN
3,Attack on Titan,"Animation, Action, Adventure",9.1,"434,457",24 min,(2013–2023),After his hometown is destroyed and his mother...,"Josh Grelle,Bryce Papenbrook,Yûki Kaji,Yui Ish...",TV-MA,NaN,434457,0,NaN
4,Jujutsu Kaisen,"Animation, Action, Adventure",8.5,"82,909",24 min,(2020– ),A boy swallows a cursed talisman - the finger ...,"Junya Enoki,Yûichi Nakamura,Adam McArthur,Yuma...",TV-MA,NaN,82909,0,NaN


In [2]:
df.shape #45,717 anime titles and 13 columns

(45717, 13)

In [3]:
df.columns.tolist()

['Title',
 'Genre',
 'User Rating',
 'Number of Votes',
 'Runtime',
 'Year',
 'Summary',
 'Stars',
 'Certificate',
 'Metascore',
 'Gross',
 'Episode',
 'Episode Title']

In [4]:
df.isnull().sum() #This counts how many missing values exist in each column.

Title                  0
Genre                  0
User Rating        20708
Number of Votes    20708
Runtime            13168
Year                 126
Summary            22170
Stars              32041
Certificate        17023
Metascore          45376
Gross              20708
Episode                0
Episode Title      10807
dtype: int64

In [5]:
df.iloc[0] # sample row

Title                                                      One Piece
Genre                                   Animation, Action, Adventure
User Rating                                                      8.9
Number of Votes                                              187,689
Runtime                                                       24 min
Year                                                        (1999– )
Summary            Follows the adventures of Monkey D. Luffy and ...
Stars              Mayumi Tanaka,Laurent Vernin,Akemi Okamura,Ton...
Certificate                                                    TV-14
Metascore                                                        NaN
Gross                                                         187689
Episode                                                            0
Episode Title                                                    NaN
Name: 0, dtype: str

In [6]:
df['Episode'].value_counts().head(10) #For our recommender we only want series-level rows (Episode = 0), not individual episodes. 
# Otherwise One Piece would appear 1000+ times.
# Run this to see how many series vs episode rows we have:

Episode
1          34909
0          10807
Episode        1
Name: count, dtype: int64

In [7]:
series_df=df[df['Episode']=='0'] # How many unique anime titles do we have at series level? — creates a True/False filter for rows where Episode is 0
print(series_df.shape) # keeps only the rows where that filter is True. We save it as series_df — our cleaner starting point
series_df.head(3)

(10807, 13)


,Title,Genre,User Rating,Number of Votes,Runtime,Year,Summary,Stars,Certificate,Metascore,Gross,Episode,Episode Title
0,One Piece,"Animation, Action, Adventure",8.9,"187,689",24 min,(1999– ),Follows the adventures of Monkey D. Luffy and ...,"Mayumi Tanaka,Laurent Vernin,Akemi Okamura,Ton...",TV-14,NaN,187689,0,NaN
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,"Animation, Action, Adventure",7.4,"28,895",99 min,(2023),The film follows the Turtle brothers as they w...,NaN,PG,74,28895,0,NaN
2,The Super Mario Bros. Movie,"Animation, Adventure, Comedy",7.1,"189,108",92 min,(2023),A plumber named Mario travels through an under...,NaN,PG,46,189108,0,NaN


``` Start cleaning ```

In [8]:
import re
# 1. Keep only series-level rows
df_clean = df[df['Episode'] == '0'].copy()

# 2. Drop columns that are too empty to be useful
df_clean = df_clean.drop(columns=['Metascore', 'Gross', 'Stars', 'Certificate', 'Summary', 'Episode', 'Episode Title'])

# 3. Clean 'Number of Votes' - remove commas, convert to number
df_clean['Number of Votes'] = df_clean['Number of Votes'].str.replace(',', '').astype(float)

# 4. Clean 'Runtime' - extract just the number
df_clean['Runtime'] = df_clean['Runtime'].str.extract(r'(\d+)').astype(float)

# 5. Clean 'Year' - extract the first 4-digit year
df_clean['Year'] = df_clean['Year'].str.extract(r'(\d{4})').astype(float)

# 6. Clean 'User Rating' - convert to number
df_clean['User Rating'] = pd.to_numeric(df_clean['User Rating'], errors='coerce')

# Check result

print(df_clean.shape)
df_clean.isnull().sum() 
    

(10807, 6)


Title                 0
Genre                 0
User Rating        3054
Number of Votes    3054
Runtime            2701
Year                145
dtype: int64

In [9]:
# Drop rows with no User Rating (we need this for the recommender)
df_clean = df_clean.dropna(subset=['User Rating'])

# Fill missing Runtime and Year with the median (middle value)
df_clean['Runtime']= df_clean['Runtime'].fillna(df_clean['Runtime'].median())
df_clean['Year'] = df_clean['Year'].fillna(df_clean['Year'].median())

# Check result
print(df_clean.shape)
df_clean.isnull().sum()

(7753, 6)


Title              0
Genre              0
User Rating        0
Number of Votes    0
Runtime            0
Year               0
dtype: int64

In [10]:
df_clean.describe()

,User Rating,Number of Votes,Runtime,Year
count,7753.000000,7.753000e+03,7753.000000,7753.000000
mean,6.805314,8.686098e+03,43.453631,2004.627112
std,0.963581,5.638453e+04,45.500280,16.217827
min,1.000000,5.000000e+00,1.000000,1907.000000
25%,6.200000,4.000000e+01,24.000000,1997.000000
50%,6.900000,2.070000e+02,25.000000,2009.000000
75%,7.500000,1.328000e+03,54.000000,2016.000000
max,9.800000,1.162284e+06,780.000000,2023.000000


Transformation

In [11]:
import numpy as np
# 1. Log-scale NUmber of Votes (reduces the huge gap between popular and obscure anime)
df_clean['Votes Log'] = np.log1p(df_clean['Number of Votes'])

# 2. Normalize User Rating and Votes Log to 0-1 scale
df_clean['Rating Norm'] = (df_clean['User Rating'] - df_clean['User Rating'].min()) / (df_clean['User Rating'].max() - df_clean['User Rating'].min())
df_clean['Votes Norm'] = (df_clean['Votes Log'] - df_clean['Votes Log'].min()) / (df_clean['Votes Log'].max() - df_clean['Votes Log'].min())

df_clean[['Title', 'User Rating', 'Rating Norm', 'Number of Votes', 'Votes Log', 'Votes Norm']].head()

                                                                                  


,Title,User Rating,Rating Norm,Number of Votes,Votes Log,Votes Norm
0,One Piece,8.9,0.897727,187689.0,12.142547,0.850227
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,7.4,0.727273,28895.0,10.271458,0.696534
2,The Super Mario Bros. Movie,7.1,0.693182,189108.0,12.150079,0.850846
3,Attack on Titan,9.1,0.920455,434457.0,12.981855,0.919169
4,Jujutsu Kaisen,8.5,0.852273,82909.0,11.325511,0.783115


Feature Engineering 

In [12]:
# 1. Popularity score - combines rating and vote count
df_clean['Popularity Score'] = 0.6 * df_clean['Rating Norm'] + 0.4 * df_clean['Votes Norm']

# 2. Split genres into a list then one-hot encode them
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

print("Genre columns created:", genre_dummies.columns.tolist())
print("Shape:", genre_dummies.shape)
genre_dummies.head(3)

Genre columns created: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War']
Shape: (7753, 22)


,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Short,Sport,Thriller,War
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Combine into final feature matrix

In [13]:
# Combine genre columns with our numeric features
feature_df = pd.concat([df_clean[['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm']].reset_index(drop=True)], axis=1)

print(feature_df.shape)
feature_df.head(3)


(7753, 4)


,Title,Popularity Score,Rating Norm,Votes Norm
0,One Piece,0.878727,0.897727,0.850227
1,Teenage Mutant Ninja Turtles: Mutant Mayhem,0.714977,0.727273,0.696534
2,The Super Mario Bros. Movie,0.756248,0.693182,0.850846


In [14]:
# Reset index on df_clean first, then rebuild
df_clean = df_clean.reset_index(drop=True)
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

feature_df = pd.concat([
    df_clean[['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm']],
    genre_dummies
], axis=1)

print(feature_df.shape)
print(feature_df.columns.tolist())

(7753, 26)
['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm', 'Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War']


Recommendation Engine

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

# Build the numeric matrix (everything except Title)
feature_matrix = feature_df.drop(columns=['Title']).values

# Compute similarity between every pair of anime
similarity_matrix = cosine_similarity(feature_matrix)

print(similarity_matrix.shape)

(7753, 7753)


In [16]:
# Fix 1: Remove duplicate titles, keep the one with most votes
df_clean = df_clean.sort_values('Number of Votes', ascending=False)
df_clean = df_clean.drop_duplicates(subset='Title', keep='first')
df_clean = df_clean.reset_index(drop=True)

# Rebuild feature matrix with deduplicated data
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

feature_df = pd.concat([
    df_clean[['Title', 'Popularity Score', 'Rating Norm', 'Votes Norm']],
    genre_dummies
], axis=1).reset_index(drop=True)

# Fix 2: Give more weight to rating and votes so similar-genre anime are ranked by quality
feature_df['Rating Norm'] = feature_df['Rating Norm'] * 3
feature_df['Votes Norm'] = feature_df['Votes Norm'] * 2

# Rebuild similarity matrix
feature_matrix = feature_df.drop(columns=['Title']).values
similarity_matrix = cosine_similarity(feature_matrix)

print(f"Unique anime: {len(df_clean)}")

Unique anime: 6808


## Fix Genre Weighting

The previous model weighted `Rating Norm * 3` and `Votes Norm * 2`, making quality the dominant signal. A typical anime has ~3 active genre columns (weight 1 each), so genre contributed only ~3 units vs ~5 for quality. Popular anime in the same broad tier cluster together regardless of genre.

Fix: weight genre columns at **3×** and bring quality features back to **1×** so genre overlap is the primary similarity driver.

In [17]:
# Inspect Sailor Moon's actual genres before fixing
print(df_clean[df_clean['Title'].str.contains('Sailor Moon', case=False)][['Title', 'Genre', 'User Rating', 'Number of Votes']])

                                                  Title  \
278                                         Sailor Moon   
736   Sailor Moon R: The Movie: The Promise of the Rose   
816                                 Sailor Moon Crystal   
883            Sailor Moon S: The Movie - Hearts in Ice   
1009    Sailor Moon SuperS: The Movie: Black Dream Hole   
1120                                Sailor Moon Eternal   
1859              Sailor Moon Super S: Ami's First Love   
2574         Bishôjo senshi Sailor Moon Super S Special   
3633                                 Sailor Moon Cosmos   
5794               Bishoujo Senshi Sailor Moon Memorial   

                             Genre  User Rating  Number of Votes  
278   Animation, Action, Adventure          7.6          12963.0  
736      Animation, Action, Comedy          7.7           2748.0  
816   Animation, Action, Adventure          7.9           2355.0  
883      Animation, Action, Comedy          7.7           2154.0  
1009     Animat

In [18]:
# Rebuild feature matrix: genres at 3x weight, quality features at 1x
genre_dummies = df_clean['Genre'].str.get_dummies(sep=', ')

# Reset rating norms to 1x (undo previous 3x/2x inflation)
rating_norm = (df_clean['User Rating'] - df_clean['User Rating'].min()) / (df_clean['User Rating'].max() - df_clean['User Rating'].min())
votes_norm = df_clean['Votes Norm'] / df_clean['Votes Norm'].max()  # re-normalize in case of drift

feature_df2 = pd.concat([
    df_clean[['Title']].reset_index(drop=True),
    (rating_norm * 1.0).rename('Rating Norm').reset_index(drop=True),
    (votes_norm * 1.0).rename('Votes Norm').reset_index(drop=True),
    (genre_dummies * 3.0).reset_index(drop=True),
], axis=1)

feature_matrix2 = feature_df2.drop(columns=['Title']).values
similarity_matrix2 = cosine_similarity(feature_matrix2)

print(f"Feature matrix shape: {feature_matrix2.shape}")
print(f"Genre weight contribution per active genre: 3.0  |  Rating/Votes: 1.0 each")

Feature matrix shape: (6808, 24)
Genre weight contribution per active genre: 3.0  |  Rating/Votes: 1.0 each


# Phase 1 — Polish & Usability

---

## Feature 1: Fuzzy Search

### The Problem
Right now the recommender uses **exact matching** — your input must match the stored title character-for-character. Miss a colon, add a space, or make a typo and it returns nothing.

```python
recommend("Naruto Shippuden")   # ✗ fails — stored as "Naruto: Shippuden"
recommend("Attack on Titen")    # ✗ fails — one letter wrong
```

This is called a **britttle interface** — it breaks on any small imperfection.

### The Concept: Edit Distance
Fuzzy search is built on a concept called **Levenshtein distance** — the minimum number of single-character edits (insert, delete, replace) needed to turn one string into another.

```
"Titen" → "Titan"  =  1 edit (replace 'e' with 'a')  → very similar
"AoT"   → "Attack on Titan"  =  many edits            → probably wrong match
```

A **similarity score** converts this distance to 0–100. Score 100 = identical. Score above ~80 = confident match.

### Real Industry Use Case
Every search bar you have ever used does this:
- **Google:** corrects "gogle" → "google"
- **Spotify:** finds "Bohemian Rapsody" despite the typo
- **GitHub search:** finds repos even with partial or misspelled names
- **E-commerce:** Amazon shows results even when you misspell a product

### The Library: `rapidfuzz`
Writing Levenshtein distance from scratch is a classic computer science algorithm — a full university lecture on **dynamic programming**. In industry, you never rewrite foundational algorithms. You use a trusted, battle-tested library. `rapidfuzz` is written in C under the hood, making it extremely fast even on thousands of comparisons.

In [19]:
from rapidfuzz import process

# --- What this function does, step by step ---
#
# 1. process.extractOne() compares your input against every title in the dataset
#    and returns the single best match, along with a confidence score (0–100)
#
# 2. We set a threshold of 80 — below that, the match is too uncertain and we
#    reject it. This prevents "AoT" from confidently matching "Azumanga Daioh".
#
# 3. If the input already matches exactly, we skip the fuzzy step entirely
#    (faster, and avoids unnecessary string comparisons).

def fuzzy_match(title, all_titles, threshold=80):
    # Step 1: try exact match first (case-insensitive)
    exact = [t for t in all_titles if t.lower() == title.lower()]
    if exact:
        return exact[0]  # found it, return immediately

    # Step 2: fuzzy match — find the closest title
    result = process.extractOne(title, all_titles)
    # result is a tuple: (matched_title, score, index)

    if result is None:
        return None

    matched_title, score, _ = result

    # Step 3: only accept the match if confidence is high enough
    if score >= threshold:
        print(f"  Did you mean: '{matched_title}'? (match score: {score:.0f}/100)")
        return matched_title
    else:
        return None  # too uncertain, reject it

# --- Demo: see fuzzy matching in action ---
all_titles = df_clean['Title'].tolist()

print("Test 1:", fuzzy_match("Naruto Shippuden", all_titles))
print("Test 2:", fuzzy_match("Attack on Titen", all_titles))
print("Test 3:", fuzzy_match("Fullmetal Alchemest", all_titles))
print("Test 4:", fuzzy_match("xyzabc123nonsense", all_titles))  # should return None

  Did you mean: 'Naruto: Shippuden'? (match score: 97/100)
Test 1: Naruto: Shippuden
  Did you mean: 'Attack on Titan'? (match score: 93/100)
Test 2: Attack on Titan
  Did you mean: 'Fullmetal Alchemist'? (match score: 95/100)
Test 3: Fullmetal Alchemist
Test 4: None


---

## Feature 2: Genre Filter

### The Problem
Right now `recommend("Attack on Titan")` always returns the 10 most similar shows by cosine similarity — no way to narrow results. But users often have a specific mood:

> *"I want something like AoT but specifically a Mystery"*
> *"Recommend me a Comedy similar to this"*

This is called **faceted filtering** — narrowing results by a specific attribute after the initial ranking. It's how Amazon's sidebar works (filter by price, brand, rating), and how Netflix's genre rows work.

### Why Filter AFTER Ranking, Not Before
You might think: just remove non-matching genres before computing similarity. But that's wrong — it would rebuild the entire similarity matrix every time.

The right approach is a **two-stage pipeline**:
1. **Stage 1 — Rank:** compute similarity scores for all 6,808 anime (done once, reused every call)
2. **Stage 2 — Filter:** after ranking, keep only results that match the requested genre

This is called a **retrieve-then-filter** pattern and is standard in search engines and recommenders. Google ranks millions of pages first, then applies your SafeSearch filter on the results — not before.

---

## Feature 3: Minimum Votes Filter

### The Problem
Your dataset has anime with as few as 5 votes. A show with 5 votes and a rating of 9.5 is not a hidden gem — it's almost certainly a data artifact (maybe 5 friends rated it). Including these pollutes your recommendations with obscure, unreliable entries.

### The Concept: Statistical Reliability
In statistics, a small **sample size** means low **confidence**. If 5 people rate something 10/10, that tells you almost nothing. If 50,000 people rate something 8.5/10, that's a meaningful signal.

This is why:
- **IMDb** requires a minimum number of votes before a film appears on their Top 250 list
- **Rotten Tomatoes** requires a minimum number of critic reviews for a Tomatometer score
- **Amazon** surfaces products with more reviews higher, even if a newer product has a slightly better average

The threshold you set is a judgment call. A common industry heuristic: require enough votes that a single person's opinion can't dominate. We'll use **50 votes** as a sensible default, but expose it as a parameter so the user can override it.

### Why a Parameter, Not a Hardcoded Number
Hardcoding `50` inside the function is called a **magic number** — it's opaque and hard to change. Exposing it as `min_votes=50` means:
- The default works for most cases
- Power users can tighten it (`min_votes=500`) or loosen it (`min_votes=10`)
- The code documents its own intent

This is a fundamental software engineering principle called **making the implicit explicit**.

---

## Feature 4: Year Range Filter

### The Problem
Anime from 1975 and anime from 2023 can share the exact same genres, but they feel completely different — different animation style, pacing, storytelling conventions, cultural context. A user saying *"recommend me something like Demon Slayer"* almost certainly doesn't want shows from 1983.

### The Concept: Temporal Relevance
Time is a powerful implicit signal in recommendation systems. It captures things that genre tags can't:
- **Visual era:** 1990s cel animation vs modern CGI
- **Narrative conventions:** episodic vs. serialized storytelling
- **Cultural moment:** what themes were popular when

Netflix uses viewing recency as a signal — shows you watched recently get weighted higher. Spotify's "Discover Weekly" deprioritizes songs from decades outside your listening patterns. They're both exploiting temporal proximity.

### Design Decision: `after` and `before` Parameters
We expose two optional parameters — `after` (minimum year) and `before` (maximum year). Using both lets users express ranges like "shows from the 2000s." Using neither means no year filter at all, preserving backwards compatibility — existing calls to the function still work exactly as before.

This pattern is called **optional, non-breaking parameters** and is how real APIs evolve over time without breaking existing users.

In [20]:
from rapidfuzz import process

def recommend(title, n=10, genre=None, min_votes=50, after=None, before=None):
    """
    Parameters
    ----------
    title     : anime title to base recommendations on (typos tolerated)
    n         : how many results to return (default 10)
    genre     : only return results that include this genre e.g. "Mystery"
    min_votes : ignore anime with fewer votes than this (default 50)
    after     : only return anime released after this year e.g. 2010
    before    : only return anime released before this year e.g. 2000
    """

    all_titles = feature_df2['Title'].tolist()

    # ----------------------------------------------------------------
    # STEP 1 — Fuzzy title matching
    # Try exact match first. If not found, use edit-distance to find
    # the closest title. Reject if confidence is below 80/100.
    # ----------------------------------------------------------------
    exact = [t for t in all_titles if t.lower() == title.lower()]
    if exact:
        matched_title = exact[0]
    else:
        result = process.extractOne(title, all_titles)
        if result is None:
            print(f"No match found for '{title}'.")
            return
        matched_title, score, _ = result
        if score < 80:
            print(f"No confident match found for '{title}'. Closest was '{matched_title}' ({score:.0f}/100).")
            return
        print(f"  Fuzzy match: '{matched_title}' (score: {score:.0f}/100)\n")

    # ----------------------------------------------------------------
    # STEP 2 — Look up similarity scores
    # Find the row index of the matched anime in our feature matrix,
    # then retrieve its precomputed similarity scores against all others.
    # We fetch extra candidates (n * 10) to leave room for filtering.
    # ----------------------------------------------------------------
    idx = feature_df2[feature_df2['Title'] == matched_title].index[0]
    all_scores = sorted(enumerate(similarity_matrix2[idx]), key=lambda x: x[1], reverse=True)
    candidates = [(i, score) for i, score in all_scores if i != idx]

    # ----------------------------------------------------------------
    # STEP 3 — Apply filters
    # Filters run on the ranked candidates in order.
    # Each filter narrows the pool without recomputing similarity.
    # ----------------------------------------------------------------
    results = []
    for i, score in candidates:
        row = df_clean.iloc[i]

        # Filter A: minimum votes — skip statistically unreliable entries
        if row['Number of Votes'] < min_votes:
            continue

        # Filter B: genre — skip if requested genre is not in this show's genres
        if genre and genre.lower() not in row['Genre'].lower():
            continue

        # Filter C: year range — skip shows outside the requested era
        if after and row['Year'] < after:
            continue
        if before and row['Year'] > before:
            continue

        results.append({
            'Title':      row['Title'],
            'Similarity': round(score, 3),
            'Rating':     round(row['User Rating'], 1),
            'Votes':      int(row['Number of Votes']),
            'Year':       int(row['Year']),
            'Genres':     row['Genre']
        })

        if len(results) == n:
            break  # we have enough, stop early

    if not results:
        print("No results matched your filters. Try relaxing the genre, year, or min_votes.")
        return

    return pd.DataFrame(results)


# ----------------------------------------------------------------
# TESTS — run all four features
# ----------------------------------------------------------------
print("=" * 60)
print("TEST 1: Fuzzy search — typo in title")
print("=" * 60)
display(recommend("Attack on Titen"))

print("=" * 60)
print("TEST 2: Fuzzy search — missing punctuation")
print("=" * 60)
display(recommend("Naruto Shippuden"))

print("=" * 60)
print("TEST 3: Genre filter — AoT but only Mystery")
print("=" * 60)
display(recommend("Attack on Titan", genre="Mystery"))

print("=" * 60)
print("TEST 4: Year filter — recent shows only (after 2015)")
print("=" * 60)
display(recommend("Attack on Titan", after=2015))

print("=" * 60)
print("TEST 5: Minimum votes filter — at least 1000 votes")
print("=" * 60)
display(recommend("Spirited Away", min_votes=1000))

print("=" * 60)
print("TEST 6: All filters combined")
print("=" * 60)
display(recommend("Attack on Titan", genre="Action", after=2010, min_votes=500))

TEST 1: Fuzzy search — typo in title
  Fuzzy match: 'Attack on Titan' (score: 93/100)



,Title,Similarity,Rating,Votes,Year,Genres
0,Fullmetal Alchemist: Brotherhood,1.0,9.1,185053,2009,"Animation, Action, Adventure"
1,One Piece,1.0,8.9,187689,1999,"Animation, Action, Adventure"
2,Princess Mononoke,1.0,8.3,414846,1997,"Animation, Action, Adventure"
3,Dragon Ball Z,1.0,8.8,140671,1989,"Animation, Action, Adventure"
4,Naruto: Shippuden,1.0,8.7,145484,2007,"Animation, Action, Adventure"
5,Cowboy Bebop,1.0,8.9,130513,1998,"Animation, Action, Adventure"
6,Hunter x Hunter,1.0,9.0,119287,2011,"Animation, Action, Adventure"
7,Demon Slayer: Kimetsu no Yaiba,1.0,8.6,131667,2019,"Animation, Action, Adventure"
8,Naruto,1.0,8.4,117447,2002,"Animation, Action, Adventure"
9,The Incredibles,1.0,8.0,777481,2004,"Animation, Action, Adventure"


TEST 2: Fuzzy search — missing punctuation
  Fuzzy match: 'Naruto: Shippuden' (score: 97/100)



,Title,Similarity,Rating,Votes,Year,Genres
0,Dragon Ball Z,1.0,8.8,140671,1989,"Animation, Action, Adventure"
1,Demon Slayer: Kimetsu no Yaiba,1.0,8.6,131667,2019,"Animation, Action, Adventure"
2,Cowboy Bebop,1.0,8.9,130513,1998,"Animation, Action, Adventure"
3,One Piece,1.0,8.9,187689,1999,"Animation, Action, Adventure"
4,Naruto,1.0,8.4,117447,2002,"Animation, Action, Adventure"
5,Hunter x Hunter,1.0,9.0,119287,2011,"Animation, Action, Adventure"
6,Fullmetal Alchemist: Brotherhood,1.0,9.1,185053,2009,"Animation, Action, Adventure"
7,Jujutsu Kaisen,1.0,8.5,82909,2020,"Animation, Action, Adventure"
8,Fullmetal Alchemist,1.0,8.5,74876,2003,"Animation, Action, Adventure"
9,Dragon Ball,1.0,8.6,63122,1995,"Animation, Action, Adventure"


TEST 3: Genre filter — AoT but only Mystery


...

,Title,Similarity,Rating,Votes,Year,Genres
0,Demon Slayer: Kimetsu no Yaiba,1.000,8.6,131667,2019,"Animation, Action, Adventure"
1,Jujutsu Kaisen,1.000,8.5,82909,2020,"Animation, Action, Adventure"
2,Vinland Saga,1.000,8.8,58698,2019,"Animation, Action, Adventure"
3,My Hero Academia,1.000,8.3,72700,2016,"Animation, Action, Adventure"
4,Incredibles 2,0.999,7.6,317085,2018,"Animation, Action, Adventure"
5,Demon Slayer the Movie: Mugen Train,0.999,8.2,67605,2020,"Animation, Action, Adventure"
6,Chainsaw Man,0.999,8.4,43330,2022,"Animation, Action, Adventure"
7,Onward,0.999,7.4,161752,2020,"Animation, Action, Adventure"
8,How to Train Your Dragon: The Hidden World,0.999,7.4,142317,2019,"Animation, Action, Adventure"
9,The Promised Neverland,0.999,8.2,43657,2019,"Animation, Action, Adventure"


TEST 5: Minimum votes filter — at least 1000 votes


,Title,Similarity,Rating,Votes,Year,Genres
0,WALL·E,1.000,8.4,1162284,2008,"Animation, Adventure, Family"
1,Howl's Moving Castle,1.000,8.2,422995,2004,"Animation, Adventure, Family"
2,The Little Mermaid,1.000,7.6,282153,1989,"Animation, Adventure, Family"
3,Castle in the Sky,1.000,8.0,174811,1986,"Animation, Adventure, Family"
4,Treasure Planet,0.999,7.2,127617,2002,"Animation, Adventure, Family"
5,Adventures of the Gummi Bears,0.998,7.5,11530,1985,"Animation, Adventure, Family"
6,Mary and the Witch's Flower,0.997,6.8,15430,2017,"Animation, Adventure, Family"
7,The Hobbit,0.997,6.7,15649,1977,"Animation, Adventure, Family"
8,Voltron: Defender of the Universe,0.997,7.9,5037,1984,"Animation, Adventure, Family"
9,Around the World with Willy Fog,0.996,7.5,3410,1983,"Animation, Adventure, Family"


TEST 6: All filters combined


,Title,Similarity,Rating,Votes,Year,Genres
0,Hunter x Hunter,1.000,9.0,119287,2011,"Animation, Action, Adventure"
1,Demon Slayer: Kimetsu no Yaiba,1.000,8.6,131667,2019,"Animation, Action, Adventure"
2,Big Hero 6,1.000,7.8,485085,2014,"Animation, Action, Adventure"
3,Jujutsu Kaisen,1.000,8.5,82909,2020,"Animation, Action, Adventure"
4,Vinland Saga,1.000,8.8,58698,2019,"Animation, Action, Adventure"
5,My Hero Academia,1.000,8.3,72700,2016,"Animation, Action, Adventure"
6,Incredibles 2,0.999,7.6,317085,2018,"Animation, Action, Adventure"
7,Demon Slayer the Movie: Mugen Train,0.999,8.2,67605,2020,"Animation, Action, Adventure"
8,Chainsaw Man,0.999,8.4,43330,2022,"Animation, Action, Adventure"
9,Onward,0.999,7.4,161752,2020,"Animation, Action, Adventure"


# Phase 2 — Better Recommendations

---

## Feature 1: Evaluation Metric (Precision@K)

**The problem:** The only way to judge recommendation quality right now is eyeballing results — subjective and unscalable.

**The fix:** Define a numeric score that measures quality objectively. We use **Precision@K** — the % of top-K results that share at least 2 genres with the input. This is the industry-standard metric for evaluating recommenders and search engines.

**Why it matters:** Without a metric, you can't tell whether a change to the model made things better or worse. Every serious ML project defines evaluation *before* tuning.

In [21]:
def precision_at_k(feature_matrix, df, k=10, sample_size=200, min_shared_genres=2, random_state=42):
    """
    Measures recommendation quality across a random sample of anime.

    For each sampled anime:
      - Get its top-k recommendations using cosine similarity
      - Count how many results share at least `min_shared_genres` genres with the input
      - That fraction is the precision score for this anime

    Final score = average precision across all sampled anime (0.0 to 1.0).

    Parameters
    ----------
    feature_matrix   : the numeric matrix used to compute similarity
    df               : the cleaned dataframe (needs 'Genre' column)
    k                : how many recommendations to evaluate (default 10)
    sample_size      : how many anime to sample for evaluation (default 200)
    min_shared_genres: minimum genre overlap to count as a good recommendation
    random_state     : seed for reproducibility — same seed = same sample every time
    """
    # --------------------------------------------------------------------------
    # STEP 1: Sample a subset of anime to evaluate
    # We don't run this on all 6,800 — that would be slow and unnecessary.
    # A random sample of 200 gives a statistically reliable estimate.
    # random_state=42 is a convention in ML (arbitrary but consistent seed)
    # that ensures you get the same sample every time — important for reproducibility.
    # --------------------------------------------------------------------------
    sample = df.sample(n=sample_size, random_state=random_state)

    # Precompute the full similarity matrix once (reused for every sampled anime)
    sim_matrix = cosine_similarity(feature_matrix)

    scores = []

    for idx in sample.index:
        # STEP 2: Get the genres of the input anime as a set
        # e.g. "Animation, Action, Adventure" → {"Animation", "Action", "Adventure"}
        input_genres = set(df.loc[idx, 'Genre'].split(', '))

        # STEP 3: Get similarity scores for this anime against all others
        # Sort descending, skip index 0 (the anime itself, always similarity=1.0)
        sim_scores = sorted(enumerate(sim_matrix[idx]), key=lambda x: x[1], reverse=True)
        top_k = [i for i, _ in sim_scores if i != idx][:k]

        # STEP 4: For each of the top-k results, check genre overlap
        hits = 0
        for result_idx in top_k:
            result_genres = set(df.loc[result_idx, 'Genre'].split(', '))
            shared = len(input_genres & result_genres)  # & = intersection (common elements)
            if shared >= min_shared_genres:
                hits += 1

        # STEP 5: Precision for this anime = hits / k
        scores.append(hits / k)

    # STEP 6: Average precision across all sampled anime
    return sum(scores) / len(scores)


# ── Baseline score ──────────────────────────────────────────────────────────
# This is the score for our current model (genre * 3, rating * 1, votes * 1).
# We save this number. Every future change must beat it to be worth keeping.

baseline_score = precision_at_k(feature_matrix2, df_clean)
print(f"Baseline Precision@10: {baseline_score:.1%}")
print()
print("How to read this: of every 10 recommendations the model makes,")
print(f"  {baseline_score*10:.1f} on average share at least 2 genres with the input.")

Baseline Precision@10: 96.0%

How to read this: of every 10 recommendations the model makes,
  9.6 on average share at least 2 genres with the input.


---

## Feature 2: Grid Search

**The problem:** Current weights (`genre * 3`, `rating * 1`, `votes * 1`) were chosen by intuition. Grid search systematically tries every combination of candidate values and picks the one with the highest Precision@10.

**Key concept:** Define a search space → evaluate all combinations → keep the winner. Simple and effective when the number of combinations is manageable (here: 60 combinations, ~1 min runtime).

**Industry note:** More sophisticated methods (Random Search, Bayesian Optimization) exist for larger search spaces, but grid search is always the right starting point — use the simplest tool that works.

In [ ]:
import itertools

# ── Define the search space ──────────────────────────────────────────────────
# These are the candidate values for each weight.
# Choosing a range around our current values (genre=3, rating=1, votes=1)
# so we explore both tighter and looser settings.

genre_weights  = [1, 2, 3, 4, 5]
rating_weights = [0.5, 1.0, 1.5, 2.0]
votes_weights  = [0.5, 1.0, 1.5]

all_combinations = list(itertools.product(genre_weights, rating_weights, votes_weights))
print(f"Total combinations to evaluate: {len(all_combinations)}")

# ── Rebuild the base features (genres + normalized rating/votes) ─────────────
# We separate them so we can rescale each independently inside the loop.

genre_dummies  = df_clean['Genre'].str.get_dummies(sep=', ')
rating_norm    = (df_clean['User Rating'] - df_clean['User Rating'].min()) / \
                 (df_clean['User Rating'].max() - df_clean['User Rating'].min())
votes_norm_raw = df_clean['Votes Norm'] / df_clean['Votes Norm'].max()

# ── Run grid search ──────────────────────────────────────────────────────────
# For each combination of weights:
#   1. Build the feature matrix with those weights applied
#   2. Score it with precision_at_k (sample_size=100 for speed during search)
#   3. Track the best-scoring combination

results = []

for gw, rw, vw in all_combinations:
    # Build feature matrix for this weight combination
    fm = pd.concat([
        (rating_norm * rw).rename('Rating Norm'),
        (votes_norm_raw * vw).rename('Votes Norm'),
        genre_dummies * gw,
    ], axis=1).values

    score = precision_at_k(fm, df_clean, sample_size=100)
    results.append({'genre_w': gw, 'rating_w': rw, 'votes_w': vw, 'precision': score})

results_df = pd.DataFrame(results).sort_values('precision', ascending=False)

print("\nTop 10 weight combinations:")
display(results_df.head(10))

# ── Best combination ─────────────────────────────────────────────────────────
best = results_df.iloc[0]
print(f"\nBest weights found:")
print(f"  genre_weight  = {best['genre_w']}")
print(f"  rating_weight = {best['rating_w']}")
print(f"  votes_weight  = {best['votes_w']}")
print(f"  Precision@10  = {best['precision']:.1%}")

Total combinations to evaluate: 60


In [ ]:
# ── Grid search conclusion ────────────────────────────────────────────────────
#
# The grid search found a large tied region — many combinations score 97%.
# Our current weights (genre=3, rating=1, votes=1) are already confirmed to
# sit inside this optimal region. There is no metric evidence to justify
# changing them. We keep them unchanged.
#
# Key principle: don't change what you can't prove needs changing.

best_gw = 3
best_rw = 1.0
best_vw = 1.0

print("Grid search conclusion: current weights are already optimal.")
print(f"  Keeping: genre={best_gw}, rating={best_rw}, votes={best_vw}")
print(f"  Baseline Precision@10: {baseline_score:.1%}")
print()
print("What grid search gave us:")
print("  - Evidence that our weights are in the right region (not just a guess)")
print("  - A reusable precision_at_k() function to measure every future change")
print("  - A lesson: when a metric hits its ceiling, it can't guide further tuning")

---

## Feature 3: Runtime Similarity

**The problem:** The model treats a 2-minute short and a 90-minute film as equally valid recommendations for each other as long as genres match. Runtime is a proxy for format — a real dimension of similarity that the model is currently blind to.

**The fix:** Add log-scaled, normalised runtime as a new feature in the matrix.

**Key ML concept:** Improving a model means identifying signals it can't currently see and encoding them as numbers. Runtime needs log-scaling first because the raw distribution is heavily skewed — most anime cluster at 24 min, with a long tail up to 780 min.

In [ ]:
import matplotlib.pyplot as plt

# ── Inspect the runtime distribution before deciding how to encode it ─────────
# Always visualise a new feature before adding it. This tells you whether
# log-scaling is needed and whether there are any obvious data problems.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw runtime distribution
axes[0].hist(df_clean['Runtime'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Runtime — Raw (minutes)')
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Count')

# Right: log-scaled runtime distribution
axes[1].hist(np.log1p(df_clean['Runtime']), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Runtime — Log-scaled')
axes[1].set_xlabel('log(1 + minutes)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Print some reference points so you can see what log-scaling does to real values
print("Reference points after log-scaling:")
for mins in [1, 5, 24, 50, 90, 120, 780]:
    print(f"  {mins:4d} min  →  log: {np.log1p(mins):.2f}")

In [ ]:
# ── Encode runtime and add it to the feature matrix ──────────────────────────

# Step 1: log-scale runtime (same technique used for votes)
runtime_log = np.log1p(df_clean['Runtime'])

# Step 2: normalise to 0–1
runtime_norm = (runtime_log - runtime_log.min()) / (runtime_log.max() - runtime_log.min())

# Step 3: rebuild feature matrix with runtime added at weight 1x
# Weight 1x keeps it as a supporting signal — genre still dominates at 3x
feature_df2 = pd.concat([
    df_clean[['Title']].reset_index(drop=True),
    (rating_norm   * 1.0).rename('Rating Norm').reset_index(drop=True),
    (votes_norm_raw * 1.0).rename('Votes Norm').reset_index(drop=True),
    (runtime_norm  * 1.0).rename('Runtime Norm').reset_index(drop=True),
    (genre_dummies * 3.0).reset_index(drop=True),
], axis=1)

feature_matrix2   = feature_df2.drop(columns=['Title']).values
similarity_matrix2 = cosine_similarity(feature_matrix2)

print(f"Feature matrix shape: {feature_matrix2.shape}")
print(f"  Previous: 24 columns (rating, votes, 22 genres)")
print(f"  Now:      25 columns (rating, votes, runtime, 22 genres)")

# ── Measure impact on Precision@10 ───────────────────────────────────────────
# If adding runtime hurts the score significantly, it's introducing noise.
# If it stays stable or improves, the feature is earning its place.

score_with_runtime = precision_at_k(feature_matrix2, df_clean, sample_size=200)

print(f"\nBaseline (no runtime):   {baseline_score:.1%}")
print(f"With runtime feature:    {score_with_runtime:.1%}")
print(f"Change:                  {score_with_runtime - baseline_score:+.1%}")

In [ ]:
# ── Qualitative check — does runtime actually change recommendations? ─────────
# Precision@10 won't catch this since it only measures genre overlap.
# Manually compare a film vs a series to verify runtime is working as expected.

print("=== Spirited Away (film, ~125 min) ===")
results = recommend("Spirited Away")
results['Runtime'] = results['Title'].map(df_clean.set_index('Title')['Runtime'])
display(results[['Title', 'Runtime', 'Rating', 'Year', 'Genres']])

print("\n=== Attack on Titan (series, 24 min/ep) ===")
results = recommend("Attack on Titan")
results['Runtime'] = results['Title'].map(df_clean.set_index('Title')['Runtime'])
display(results[['Title', 'Runtime', 'Rating', 'Year', 'Genres']])

---

## Feature 4: Era Similarity

**The problem:** Two shows can share identical genre tags but feel completely different because one is from 1986 and the other from 2020 — different animation style, pacing, and narrative conventions. The model is currently blind to this.

**The fix:** Add normalised release year as a soft signal so temporally close shows score higher. This complements the existing hard year filters (`after=`, `before=`) — those exclude shows entirely; this nudges scores continuously.

**Why no log-scaling:** Year ranges 1907–2023 (116 years, no extreme outliers). Min-max normalisation is sufficient — log-scaling is only needed when a feature has a long tail of extreme values like votes (232,000× range).

In [ ]:
# ── Inspect year distribution ─────────────────────────────────────────────────
# Always visualise before encoding. Check for skew, outliers, and data quality.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_clean['Year'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Release Year — Distribution')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

# Zoom into the meaningful range (post-1960) to see the real shape
axes[1].hist(df_clean[df_clean['Year'] >= 1960]['Year'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Release Year — Post-1960 (zoomed)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Year range: {int(df_clean['Year'].min())} – {int(df_clean['Year'].max())}")
print(f"Pre-1960 titles: {(df_clean['Year'] < 1960).sum()} ({(df_clean['Year'] < 1960).mean():.1%} of dataset)")

In [ ]:
# ── Encode year and add to feature matrix ────────────────────────────────────

# Min-max normalise year to 0–1
# 1907 → 0.0  (oldest)   2023 → 1.0  (newest)
year_norm = (df_clean['Year'] - df_clean['Year'].min()) / \
            (df_clean['Year'].max() - df_clean['Year'].min())

# Rebuild feature matrix with year added at weight 1x
feature_df2 = pd.concat([
    df_clean[['Title']].reset_index(drop=True),
    (rating_norm    * 1.0).rename('Rating Norm').reset_index(drop=True),
    (votes_norm_raw * 1.0).rename('Votes Norm').reset_index(drop=True),
    (runtime_norm   * 1.0).rename('Runtime Norm').reset_index(drop=True),
    (year_norm      * 1.0).rename('Year Norm').reset_index(drop=True),
    (genre_dummies  * 3.0).reset_index(drop=True),
], axis=1)

feature_matrix2    = feature_df2.drop(columns=['Title']).values
similarity_matrix2 = cosine_similarity(feature_matrix2)

print(f"Feature matrix shape: {feature_matrix2.shape}")
print(f"  Previous: 25 columns (rating, votes, runtime, 22 genres)")
print(f"  Now:      26 columns (rating, votes, runtime, year, 22 genres)")

# ── Measure impact ────────────────────────────────────────────────────────────
score_with_year = precision_at_k(feature_matrix2, df_clean, sample_size=200)

print(f"\nBaseline (no year):   {baseline_score:.1%}")
print(f"With year feature:    {score_with_year:.1%}")
print(f"Change:               {score_with_year - baseline_score:+.1%}")

In [ ]:
# ── Qualitative check — does era grouping work? ───────────────────────────────
# Compare a 1986 classic vs a 2019 modern show.
# We expect 1986 results to skew older, 2019 results to skew newer.

for title in ["Dragon Ball Z", "Demon Slayer: Kimetsu no Yaiba"]:
    print(f"=== {title} ===")
    results = recommend(title)
    results['Year'] = results['Title'].map(df_clean.set_index('Title')['Year'].astype(int))
    results['Runtime'] = results['Title'].map(df_clean.set_index('Title')['Runtime'].astype(int))
    display(results[['Title', 'Year', 'Runtime', 'Rating', 'Genres']])
    print(f"  Mean recommendation year: {results['Year'].mean():.0f}\n")

# Phase 3 — Understanding What Shows Are About

---

## Feature 1: TF-IDF on Plot Summaries

**The problem:** The model knows genre tags but not plot content. Two shows can share `Action, Adventure` and be completely different — Attack on Titan is a dark survival story inside giant walls; One Piece is a lighthearted pirate adventure. Genre tags are too coarse to capture this.

**The fix:** Use TF-IDF (Term Frequency – Inverse Document Frequency) to turn each show's plot summary into a numeric vector. Words that are distinctive to a summary (like "mecha", "demon", "pirate") get high scores; common filler words ("the", "a", "is") are automatically ignored.

**Why 300 dimensions:** TF-IDF creates one column per word in the vocabulary — potentially 5,000–10,000 columns. We keep only the top 300 most informative words to avoid overloading the matrix with noise.

**Why weight 0.5:** Genre columns contribute ~sqrt(27) ≈ 5.2 to the feature vector (3 active genres × weight 3). TF-IDF is L2-normalised (unit length) then multiplied by 0.5, so it contributes 0.5. This keeps TF-IDF as a supporting signal that breaks ties between genre-similar shows, without overriding genre as the primary driver.

In [ ]:
# ── Re-attach summaries ────────────────────────────────────────────────────────
# We dropped the Summary column during cleaning. Re-attach it from the original
# df using Title as the key. drop_duplicates handles the fact that df has one
# row per episode — we only want one summary per series title.

summary_map = df.drop_duplicates(subset='Title', keep='first').set_index('Title')['Summary']
df_clean['Summary'] = df_clean['Title'].map(summary_map).fillna('')

# ── Coverage check ────────────────────────────────────────────────────────────
# ~22% of series had no summary in the original dataset (see isnull check above).
# fillna('') means those rows contribute nothing to TF-IDF — not penalised, just neutral.

n_with = (df_clean['Summary'] != '').sum()
n_total = len(df_clean)
print(f"Summaries available: {n_with} / {n_total} ({n_with/n_total:.1%})")
print()
print("Sample summaries:")
for title in ["Attack on Titan", "Spirited Away", "Neon Genesis Evangelion", "One Piece"]:
    row = df_clean[df_clean['Title'] == title]
    if len(row) > 0:
        s = row.iloc[0]['Summary']
        print(f"  {title}: {s[:100]}...")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Fit TF-IDF on all summaries ───────────────────────────────────────────────
# max_features=300 : keep only the 300 most informative words
# stop_words='english' : automatically ignore "the", "a", "is", "of", etc.
# norm='l2' (default) : each anime's TF-IDF vector has L2 norm = 1.0

tfidf = TfidfVectorizer(max_features=300, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_clean['Summary']).toarray()

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Sample vocabulary: {tfidf.get_feature_names_out()[:20].tolist()}")
print()

# ── Scale and rebuild feature matrix ─────────────────────────────────────────
tfidf_weight = 0.5
tfidf_df = pd.DataFrame(
    tfidf_matrix * tfidf_weight,
    columns=[f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
)

feature_df2 = pd.concat([
    df_clean[['Title']].reset_index(drop=True),
    (rating_norm    * 1.0).rename('Rating Norm').reset_index(drop=True),
    (votes_norm_raw * 1.0).rename('Votes Norm').reset_index(drop=True),
    (runtime_norm   * 1.0).rename('Runtime Norm').reset_index(drop=True),
    (year_norm      * 1.0).rename('Year Norm').reset_index(drop=True),
    (genre_dummies  * 3.0).reset_index(drop=True),
    tfidf_df.reset_index(drop=True),
], axis=1)

feature_matrix2    = feature_df2.drop(columns=['Title']).values
similarity_matrix2 = cosine_similarity(feature_matrix2)

print(f"Feature matrix shape: {feature_matrix2.shape}")
print(f"  Previous: 26 columns (rating, votes, runtime, year, 22 genres)")
print(f"  Now:      {feature_matrix2.shape[1]} columns (+300 TF-IDF dimensions)")

# ── Measure impact on Precision@10 ───────────────────────────────────────────
score_with_tfidf = precision_at_k(feature_matrix2, df_clean, sample_size=200)

print(f"\nBaseline (no TF-IDF):  {baseline_score:.1%}")
print(f"With TF-IDF:           {score_with_tfidf:.1%}")
print(f"Change:                {score_with_tfidf - baseline_score:+.1%}")
print()
print("Note: Precision@10 measures genre overlap — it cannot measure plot similarity.")
print("A flat score here means TF-IDF didn't hurt genre recommendations.")
print("The real value shows up in the qualitative check below.")

In [ ]:
# ── Qualitative check ─────────────────────────────────────────────────────────
# Precision@10 only measures genre overlap — it can't tell us whether TF-IDF
# is surfacing content-similar shows. Check manually with two shows that share
# the same genres but have very different plots.
#
# Neon Genesis Evangelion: psychological breakdown, existential dread, mecha pilots
# One Piece: pirate crew searching for treasure, friendship and adventure
#
# Both are Action/Adventure. Genre alone cannot distinguish them.
# With TF-IDF, plot words ("pilot", "angel", "AT Field" vs "pirate", "fruit", "navy")
# should push their recommendation lists apart.

print("=== Neon Genesis Evangelion ===")
nge = recommend("Neon Genesis Evangelion")
nge['Summary_preview'] = nge['Title'].map(
    df_clean.set_index('Title')['Summary'].str[:80]
)
display(nge[['Title', 'Rating', 'Year', 'Genres']])

print("\n=== One Piece ===")
op = recommend("One Piece")
op['Summary_preview'] = op['Title'].map(
    df_clean.set_index('Title')['Summary'].str[:80]
)
display(op[['Title', 'Rating', 'Year', 'Genres']])